# Held–Suarez (1994) production run — Palintropos T42 L20, 1200 days

Launcher for the frozen experiment of `docs/held_suarez/PROTOCOL.md` and
`docs/superpowers/plans/2026-09-27-held-suarez-24h-plan.md` (stages S7–S10).
Every numerical step goes through **one entrypoint**,
`python -m tropoi.run.held_suarez` (`run`, `benchmark`, `report`, `verify`) at the
pinned `SOLVER_COMMIT`; this notebook only chooses directories, checks the GPU,
and calls it.

**Human steps (S7):** set the parameters in the next cell, then *Runtime → Run all*.
Colab asks for Google Drive consent once.

What happens, in order:

1. pinned checkout + dependency check;
2. **hard GPU / float64 capability check** (stops on failure);
3. Drive mount and explicit run / backup directories;
4. Dinosaur reference: located on Drive and hash-verified (never recomputed here);
5. **benchmark gate** (S8): 200 complete steps including the normal daily statistics
   and one atomic checkpoint write; projects the 1200-day wall time and **stops with
   `DECISION NEEDED`** if it exceeds `MAX_RUN_HOURS` or `DEADLINE`. Resolution is
   never lowered automatically;
6. restore from the Drive backup if this Colab VM has no local run (resume case);
7. the fixed-Δt run loop (resumes from the newest checkpoint, **never repeats the RK4
   startup**), Drive backup every 10 simulated days keeping 3;
8. the deterministic tier A/B/C report.

**Resuming after a disconnect:** open the notebook again, keep the same parameters, and
*Run all*. The benchmark is skipped automatically when a run already exists, the
latest backup is restored and hash-verified, and the run continues from its last
checkpoint with both leapfrog time levels (no new startup, same configuration hash;
a different commit or configuration is refused).

In [ ]:
# ---- parameters (edit before Run all) --------------------------------------
import os
import pathlib
import sys

SOLVER_COMMIT = "21203ff565eb09996eade1de4f8f1f30ce551ab0"   # immutable commit containing the solver; must be pushed to origin
REPO_URL = "https://github.com/AlexandreEros/Palintropos.git"

PRESET = "production"                  # evolved T42 (store 43, retain 42), L20, 64x128, 1200 days (PROTOCOL.md §2)
EXPERIMENT_ID = "hs-T42L20-prod-001"
DT_SECONDS = 900.0                     # frozen at launch (DEVLOG 2026-09-27 S4b)

DRIVE_ROOT = "/content/drive/MyDrive/Palintropos-HS"   # synced locally to G:\O meu disco\Palintropos-HS
LOCAL_ROOT = "/content/hs"                                # fast VM disk for the live run

MAX_RUN_HOURS = 24.0                   # allowed compute for the whole 1200-day run (sessions chain by resume)
DEADLINE = "2026-09-28T07:25:00Z"      # plan T0 + 24 h; set None to accept finishing later (plan §6.5)
SESSION_HOURS = 23.5                   # stop cleanly (checkpoint + backup) before the Colab session ends
BENCHMARK_STEPS = 200

REFERENCE_SERIES_SHA256 = "baa7cffd59f833d4560f23de426a2932ca574744ee4afc6df68ad60034cc4663"

# Local smoke of this notebook (same entrypoint, T21 L10, 2 days, no Drive, no clone):
LOCAL_SMOKE = os.environ.get("HS_NOTEBOOK_LOCAL_SMOKE") == "1"
if LOCAL_SMOKE:
    PRESET, EXPERIMENT_ID, DT_SECONDS = "smoke", "hs-T21L10-smoke", 1200.0
    LOCAL_ROOT = os.environ["HS_NOTEBOOK_WORK"]
    DRIVE_ROOT = os.path.join(LOCAL_ROOT, "drive")
    MAX_RUN_HOURS, DEADLINE, SESSION_HOURS = 1.0, None, 1.0
SMOKE_FORCED_STOP_DAY = 1 if LOCAL_SMOKE else None   # smoke only: stop at day 1, then resume
print(f"{EXPERIMENT_ID}: preset={PRESET} dt={DT_SECONDS} s commit={SOLVER_COMMIT}")

In [ ]:
# ---- 1. pinned checkout + dependencies --------------------------------------
import subprocess

if LOCAL_SMOKE:
    SRC = pathlib.Path(os.environ["HS_NOTEBOOK_SRC"])
else:
    SRC = pathlib.Path("/content/palintropos-src")
    if not (SRC / ".git").exists():
        subprocess.check_call(["git", "clone", "--quiet", REPO_URL, str(SRC)])
    subprocess.check_call(["git", "-C", str(SRC), "fetch", "--quiet", "origin"])
    subprocess.check_call(["git", "-C", str(SRC), "checkout", "--quiet", "--detach", SOLVER_COMMIT])
head = subprocess.check_output(["git", "-C", str(SRC), "rev-parse", "HEAD"], text=True).strip()
if not LOCAL_SMOKE:
    assert head == SOLVER_COMMIT, f"checkout produced {head}, expected {SOLVER_COMMIT}"
    dirty = subprocess.check_output(["git", "-C", str(SRC), "status", "--porcelain"], text=True)
    assert not dirty.strip(), f"pinned checkout is not clean:\n{dirty}"
print("solver source:", SRC, "HEAD", head)

# Colab ships numpy/scipy/matplotlib and a CUDA-12 CuPy; install only what is missing
# (requirements.txt pins the local CUDA-11 CuPy). The package is NOT pip-installed: it runs
# from SRC via PYTHONPATH so that its provenance check sees the pinned git checkout.
if sys.version_info < (3, 12):
    raise SystemExit(f"STOP: Python >= 3.12 required, this runtime has {sys.version}")
import importlib.util
if importlib.util.find_spec("cupy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "cupy-cuda12x"])
ENV = dict(os.environ, PYTHONPATH=str(SRC / "src"), PYTHONUNBUFFERED="1")
HS = [sys.executable, "-m", "tropoi.run.held_suarez"]
CFG_ARGS = ["--preset", PRESET, "--experiment-id", EXPERIMENT_ID, "--dt", str(DT_SECONDS)]
out = subprocess.run(HS + ["show-config"] + CFG_ARGS, env=ENV, cwd=SRC, check=True,
                     capture_output=True, text=True).stdout
import json
CONFIG = json.loads(out)
print("config_sha256", CONFIG["config_sha256"], "| total steps", CONFIG["total_steps"],
      "| effective truncation T%d" % CONFIG["effective_truncation"])

In [ ]:
# ---- 2. hard GPU / float64 capability check ---------------------------------
import numpy as np
import cupy as cp

if not cp.is_available():
    raise SystemExit("STOP: no CUDA GPU. Runtime -> Change runtime type -> GPU.")
dev = cp.cuda.Device()
props = cp.cuda.runtime.getDeviceProperties(dev.id)
name = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
cc = dev.compute_capability
rng = np.random.default_rng(0)
a = rng.standard_normal((512, 512)); b = rng.standard_normal((512, 512))
c_gpu = cp.asnumpy(cp.asarray(a) @ cp.asarray(b))
err = float(np.abs(c_gpu - a @ b).max() / np.abs(a @ b).max())
z = cp.asarray(a, dtype=cp.float64)
if z.dtype != cp.float64 or err > 1e-12:
    raise SystemExit(f"STOP: float64 GEMM check failed on {name} (relative error {err:.2e})")
tiny = float(cp.asnumpy(cp.asarray(1.0, dtype=cp.float64) + cp.asarray(1e-15, dtype=cp.float64)) - 1.0)
if not (0.5e-15 < tiny < 2e-15):
    raise SystemExit(f"STOP: device arithmetic is not IEEE float64 ({tiny})")
free, total = cp.cuda.runtime.memGetInfo()
print(f"GPU OK: {name}, compute capability {cc}, float64 GEMM rel. error {err:.1e}, "
      f"memory {total / 2**30:.1f} GiB, CuPy {cp.__version__}, "
      f"CUDA runtime {cp.cuda.runtime.runtimeGetVersion()}")

In [ ]:
# ---- 3. Drive mount and run directories -------------------------------------
if not LOCAL_SMOKE:
    from google.colab import drive
    drive.mount("/content/drive")
RUN_DIR = pathlib.Path(LOCAL_ROOT) / EXPERIMENT_ID          # live run (fast local disk)
BACKUP_DIR = pathlib.Path(DRIVE_ROOT) / EXPERIMENT_ID       # every 10 days, newest 3 kept
REPORT_DIR = BACKUP_DIR / "report"
BENCH_DIR = pathlib.Path(LOCAL_ROOT) / "benchmark"
for d in (RUN_DIR.parent, BACKUP_DIR, BENCH_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("run dir   :", RUN_DIR)
print("backup dir:", BACKUP_DIR)

In [ ]:
# ---- 4. Dinosaur reference (T42 L20 float64, 1200 days; computed separately) ----
# Produced by tools/held_suarez/dino_reference.py (run locally on CPU, 2026-09-27, or in a
# second Colab session). Copy runs/hs-reference-dinosaur-T42L20-001/series.npz (and its
# b2.json, see docs/held_suarez/STATUS.md) to <DRIVE_ROOT>/reference/. Never recomputed here.
import hashlib
REFERENCE_DIR = pathlib.Path(DRIVE_ROOT) / "reference" / "hs-reference-dinosaur-T42L20-001"
if LOCAL_SMOKE:
    REFERENCE_DIR = SRC / "runs" / "hs-reference-dinosaur-T42L20-001"
REFERENCE = REFERENCE_DIR / "series.npz"
REFERENCE_B2 = REFERENCE_DIR / "b2.json"
if not REFERENCE_B2.exists():
    # the committed B2 evidence for exactly this reference series (hash-checked)
    committed = SRC / "docs" / "held_suarez" / "REFERENCE_B2.json"
    if json.loads(committed.read_text())["series_sha256"] == REFERENCE_SERIES_SHA256:
        REFERENCE_B2 = committed
print("reference B2 evidence:", REFERENCE_B2)
if REFERENCE.exists():
    h = hashlib.sha256(REFERENCE.read_bytes()).hexdigest()
    if h != REFERENCE_SERIES_SHA256:
        raise SystemExit(f"STOP: reference series hash {h} != pinned {REFERENCE_SERIES_SHA256}")
    print("reference verified:", REFERENCE)
else:
    print("WARNING: no reference at", REFERENCE, "- tier C will be INCONCLUSIVE until it is provided.")
    print("To produce it (second session, CPU is enough, ~5 h):")
    print("  pip install 'git+https://github.com/neuralgcm/dinosaur.git@be5409da2636918cd602ee75b201de1617f3b3aa'")
    print("  python tools/held_suarez/dino_reference.py --out <REFERENCE_DIR> --days 1200")

In [ ]:
# ---- 5. benchmark gate (S8) --------------------------------------------------
existing = sorted((RUN_DIR / "checkpoints").glob("checkpoint-s*.npz")) + \
    sorted((BACKUP_DIR / "checkpoints").glob("checkpoint-s*.npz"))
if existing and not LOCAL_SMOKE:
    print(f"run already started ({existing[-1].name}); benchmark gate skipped on resume")
else:
    cmd = HS + ["benchmark"] + CFG_ARGS + ["--work-dir", str(BENCH_DIR),
                                           "--steps", str(BENCHMARK_STEPS),
                                           "--max-hours", str(MAX_RUN_HOURS)]
    if DEADLINE:
        cmd += ["--deadline", DEADLINE]
    if LOCAL_SMOKE:
        cmd += ["--allow-dirty"]
    r = subprocess.run(cmd, env=ENV, cwd=SRC, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode == 3:
        raise SystemExit("DECISION NEEDED: benchmark projects past the allowed wall time or "
                         "deadline (see above). Not launching. Options: bigger GPU, later deadline, "
                         "faster transforms. The resolution is not lowered automatically.")
    r.check_returncode()

In [ ]:
# ---- 6. restore from the Drive backup (resume on a fresh VM) -----------------
import shutil
local = sorted((RUN_DIR / "checkpoints").glob("checkpoint-s*.npz"))
backup = sorted((BACKUP_DIR / "checkpoints").glob("checkpoint-s*.npz"))
if not local and backup:
    (RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
    for name in ("config.json", "events.jsonl", "series.json", "statistics.npz", "progress.json"):
        if (BACKUP_DIR / name).exists():
            shutil.copy2(BACKUP_DIR / name, RUN_DIR / name)
    for cand in reversed(backup):                  # newest verifiable backup wins
        target = RUN_DIR / "checkpoints" / cand.name
        shutil.copy2(cand, target)
        ok = subprocess.run(HS + ["verify", "--run-dir", str(RUN_DIR)], env=ENV, cwd=SRC).returncode == 0
        if ok:
            print("restored", cand.name, "from", BACKUP_DIR)
            break
        print("backup", cand.name, "failed verification; trying an older one")
        target.unlink()
    else:
        raise SystemExit("STOP: no Drive backup passes hash verification; inspect " + str(BACKUP_DIR))
if (RUN_DIR / "checkpoints").exists():
    subprocess.run(HS + ["verify", "--run-dir", str(RUN_DIR)], env=ENV, cwd=SRC, check=True)

In [ ]:
# ---- 7. the run (fixed dt, resumable, Drive backup every 10 days, keep 3) -----
def run_segment(until_day=None):
    cmd = HS + ["run"] + CFG_ARGS + ["--run-dir", str(RUN_DIR), "--backup-dir", str(BACKUP_DIR),
                                     "--backup-every-days", "10",
                                     "--max-wall-hours", str(SESSION_HOURS)]
    if until_day is not None:
        cmd += ["--until-day", str(until_day)]
    if LOCAL_SMOKE:
        cmd += ["--allow-dirty", "--backup-every-days", "1"]
    p = subprocess.Popen(cmd, env=ENV, cwd=SRC, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc == 2:
        raise SystemExit("REFUSED: configuration/commit mismatch or corrupt checkpoint (see above)")
    if rc == 4:
        raise SystemExit("ABORTED: validation failure (tier B1); the report records it")
    if rc != 0:
        raise SystemExit(f"run exited with {rc}")

if SMOKE_FORCED_STOP_DAY is not None:
    run_segment(SMOKE_FORCED_STOP_DAY)     # forced stop, then a fresh process resumes
run_segment()
progress = json.loads((RUN_DIR / "progress.json").read_text())
print("progress:", progress)
if progress["day"] < progress["days_total"]:
    print(f"RUN NOT FINISHED (day {progress['day']:.2f} of {progress['days_total']}): the session "
          "limit was reached after a checkpoint + backup. Open the notebook in a new session and "
          "Run all to resume (no new startup). The report below is INCOMPLETE by design.")

In [ ]:
# ---- 8. deterministic tier A/B/C report --------------------------------------
cmd = HS + ["report", "--run-dir", str(RUN_DIR), "--out", str(REPORT_DIR)]
if REFERENCE.exists():
    cmd += ["--reference", str(REFERENCE)]
if REFERENCE_B2.exists():
    cmd += ["--reference-b2", str(REFERENCE_B2)]
r = subprocess.run(cmd, env=ENV, cwd=SRC, capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-2000:])
r.check_returncode()